In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV


Data = pd.read_csv("../data/Titanic-Dataset.csv")
print(Data.head())
print(Data.describe())
print(Data.info())
print(Data.isna().sum())

# visualizing the missing values from the dataset
sns.heatmap(Data.isna(), cbar=False)
plt.title("Heat Map of missing Values")
plt.show()


#dropping the PassengerId and Cabin columns
Data = Data.drop(columns=["PassengerId", "Ticket"])
# feature engineering
Data["Title"] = Data["Name"].str.extract(r",\s*([^\.]+)\.")
Data["Deck"] = Data["Cabin"].str[0]
Data["Deck"]= Data["Deck"].fillna("Missing")
Data = Data.drop(columns=["Name", "Cabin"])
Data["Age"]= Data["Age"].fillna(Data["Age"].median())
Data["Fare"] = Data["Fare"].fillna(Data["Fare"].median())

print(Data.head())
print(Data.describe())
print(Data.info())
print(Data.isna().sum()) 

# seperating the features according to the data types

X = Data.drop(columns=["Survived"])
y = Data["Survived"]

num_cols = X.select_dtypes(include=["int64","float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# visualizing data

for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=Data, x="Survived", y=col)
    plt.title(f"{col} vs Survived")
    plt.xlabel("1 = Survived, 0 = Dead)")
    plt.show()


plt.figure(figsize=(10,9))
corr = Data[num_cols.tolist() + ["Survived"]].corr()
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Map")
plt.show()

#spliting the dataset into train and test subsets 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# preprocessing
num_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])


model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))

])


rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    random_state=42,
    class_weight="balanced"
))
])

param_grid = {
    "model__n_estimators":[100, 300, 500],
    "model__max_depth": [3,5,8],
    "model__min_samples_leaf": [1, 3, 5]
}

grid = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring="recall",
    n_jobs=1
)

grid.fit(X_train,y_train)

print("Best Params", grid.best_params_)
print("Best score", grid.best_score_)

rf_best = grid.best_estimator_
y_pred_rf = rf_best.predict(X_test)
print("Classification report for random forest after using best estimator",classification_report(y_test, y_pred_rf))

rf_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="recall")

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="recall"
)


model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("classification report for LogReg",classification_report(y_test, y_pred))

print("Logistic regression Mean CV score", cv_scores.mean())
print("Logistic Regression cv std score", cv_scores.std())

print("Rf mean score", rf_scores.mean())

print("Rf std score", rf_scores.std())
print(Data["Survived"].value_counts())

# 




classification report for LogReg               precision    recall  f1-score   support

           0       0.88      0.83      0.85       110
           1       0.75      0.81      0.78        69

    accuracy                           0.82       179
   macro avg       0.81      0.82      0.81       179
weighted avg       0.83      0.82      0.82       179

classification report for tuned LogReg               precision    recall  f1-score   support

           0       0.90      0.67      0.77       110
           1       0.63      0.88      0.73        69

    accuracy                           0.75       179
   macro avg       0.77      0.78      0.75       179
weighted avg       0.80      0.75      0.76       179

Logistic regression Mean CV score 0.8024242424242425
Logistic Regression cv std score 0.04899396005244157
Rf mean score 0.7109764309764309
Rf std score 0.07384249076202017
Survived
0    549
1    342
Name: count, dtype: int64
